In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression
from typing import Optional
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
import optuna




In [ ]:
df = pd.read_csv("/content/drive/MyDrive/model_price_car/train.csv")
df

,id,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,0,MINI,Cooper S Base,2007,213000,Gasoline,172.0HP 1.6L 4 Cylinder Engine Gasoline Fuel,A/T,Yellow,Gray,None reported,Yes,4200
1,1,Lincoln,LS V8,2002,143250,Gasoline,252.0HP 3.9L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Beige,At least 1 accident or damage reported,Yes,4999
2,2,Chevrolet,Silverado 2500 LT,2002,136731,E85 Flex Fuel,320.0HP 5.3L 8 Cylinder Engine Flex Fuel Capab...,A/T,Blue,Gray,None reported,Yes,13900
3,3,Genesis,G90 5.0 Ultimate,2017,19500,Gasoline,420.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Black,Black,None reported,Yes,45000
4,4,Mercedes-Benz,Metris Base,2021,7388,Gasoline,208.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,7-Speed A/T,Black,Beige,None reported,Yes,97500
...,...,...,...,...,...,...,...,...,...,...,...,...,...
188528,188528,Cadillac,Escalade ESV Platinum,2017,49000,Gasoline,420.0HP 6.2L 8 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,White,Beige,None reported,Yes,27500
188529,188529,Mercedes-Benz,AMG C 43 AMG C 43 4MATIC,2018,28600,Gasoline,385.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,At least 1 accident or damage reported,Yes,30000
188530,188530,Mercedes-Benz,AMG GLC 63 Base 4MATIC,2021,13650,Gasoline,469.0HP 4.0L 8 Cylinder Engine Gasoline Fuel,7-Speed A/T,White,Black,None reported,Yes,86900
188531,188531,Audi,S5 3.0T Prestige,2022,13895,Gasoline,3.0L,1-Speed Automatic,Daytona Gray Pearl Effect,Black,None reported,NaN,84900


добавить фичу: страна,

In [ ]:
def parse_engine(feature_str):
    # Извлекаем мощность
    power_match = re.search(r'(\d+\.?\d*)HP', feature_str)
    power = float(power_match.group(1)) if power_match else None

    # Извлекаем объем двигателя (как float)
    displacement_match = re.search(r'(\d+\.?\d*)L', feature_str)
    displacement = float(displacement_match.group(1)) if displacement_match else None

    # Извлекаем количество цилиндров (как int)
    cylinders = None
    cylinders_match = re.search(r'(\d+)\s*Cylinder', feature_str)
    if cylinders_match:
        cylinders = int(cylinders_match.group(1))
    else:
        v_match = re.search(r'V(\d+)', feature_str)
        if v_match:
            cylinders = int(v_match.group(1))



    # Возвращаем ТОЛЬКО 4 значения: мощность, объем, цилиндры, топливо
    return pd.Series([power, displacement, cylinders])

# Применяем функцию (только 4 колонки)
df[['power_hp', 'engine_volume_l', 'cylinders_count']] = df['engine'].apply(parse_engine)


In [ ]:
df = df.drop('engine', axis=1)

In [ ]:
df['accident'] = df['accident'].apply(lambda x: 1 if pd.notna(x) and x != "None reported" else 0)
df

,id,brand,model,model_year,milage,fuel_type,transmission,ext_col,int_col,accident,clean_title,price,power_hp,engine_volume_l,cylinders_count
0,0,MINI,Cooper S Base,2007,213000,Gasoline,A/T,Yellow,Gray,0,Yes,4200,172.0,1.6,4.0
1,1,Lincoln,LS V8,2002,143250,Gasoline,A/T,Silver,Beige,1,Yes,4999,252.0,3.9,8.0
2,2,Chevrolet,Silverado 2500 LT,2002,136731,E85 Flex Fuel,A/T,Blue,Gray,0,Yes,13900,320.0,5.3,8.0
3,3,Genesis,G90 5.0 Ultimate,2017,19500,Gasoline,Transmission w/Dual Shift Mode,Black,Black,0,Yes,45000,420.0,5.0,8.0
4,4,Mercedes-Benz,Metris Base,2021,7388,Gasoline,7-Speed A/T,Black,Beige,0,Yes,97500,208.0,2.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188528,188528,Cadillac,Escalade ESV Platinum,2017,49000,Gasoline,Transmission w/Dual Shift Mode,White,Beige,0,Yes,27500,420.0,6.2,8.0
188529,188529,Mercedes-Benz,AMG C 43 AMG C 43 4MATIC,2018,28600,Gasoline,8-Speed A/T,White,Black,1,Yes,30000,385.0,3.0,6.0
188530,188530,Mercedes-Benz,AMG GLC 63 Base 4MATIC,2021,13650,Gasoline,7-Speed A/T,White,Black,0,Yes,86900,469.0,4.0,8.0
188531,188531,Audi,S5 3.0T Prestige,2022,13895,Gasoline,1-Speed Automatic,Daytona Gray Pearl Effect,Black,0,NaN,84900,NaN,3.0,NaN


In [ ]:
def parse_transmission(trans_str):
    if pd.isna(trans_str):
        return pd.Series([None, None])

    trans_str = str(trans_str)

    # 1. Определяем тип (автомат или механика)
    is_automatic = 1  # по умолчанию автомат

    # Признаки механики
    manual_keywords = ['M/T', 'Manual']
    if any(keyword in trans_str for keyword in manual_keywords):
        is_automatic = 0

    # Особые случаи
    if 'CVT' in trans_str:  # Вариатор - тоже автомат
        is_automatic = 1

    # 2. Извлекаем количество скоростей
    # Ищем цифру перед словами Speed, A/T, M/T, Automatic
    speed_match = re.search(r'(\d+)-?\s*(Speed|A/T|M/T|Automatic)', trans_str)

    if speed_match:
        speeds = int(speed_match.group(1))
    elif 'A/T' in trans_str and 'Speed' not in trans_str:
        # Просто "A/T" - без указания скоростей
        speeds = 4  # или None, или 5 - зависит от данных
    elif 'M/T' in trans_str and 'Speed' not in trans_str:
        # Просто "M/T" - механика без указания скоростей
        speeds = 5  # обычно 5-ступка
    elif 'Automatic' in trans_str and 'Speed' not in trans_str:
        # Просто "Automatic" без цифры
        speeds = 4
    elif 'CVT' in trans_str:
        speeds = 0  # CVT не имеет фиксированных скоростей
    else:
        speeds = None

    return pd.Series([is_automatic, speeds])

# Применяем
df[['is_automatic', 'gears_count']] = df['transmission'].apply(parse_transmission)
df

,id,brand,model,model_year,milage,fuel_type,transmission,ext_col,int_col,accident,clean_title,price,power_hp,engine_volume_l,cylinders_count,is_automatic,gears_count
0,0,MINI,Cooper S Base,2007,213000,Gasoline,A/T,Yellow,Gray,0,Yes,4200,172.0,1.6,4.0,1.0,4.0
1,1,Lincoln,LS V8,2002,143250,Gasoline,A/T,Silver,Beige,1,Yes,4999,252.0,3.9,8.0,1.0,4.0
2,2,Chevrolet,Silverado 2500 LT,2002,136731,E85 Flex Fuel,A/T,Blue,Gray,0,Yes,13900,320.0,5.3,8.0,1.0,4.0
3,3,Genesis,G90 5.0 Ultimate,2017,19500,Gasoline,Transmission w/Dual Shift Mode,Black,Black,0,Yes,45000,420.0,5.0,8.0,1.0,NaN
4,4,Mercedes-Benz,Metris Base,2021,7388,Gasoline,7-Speed A/T,Black,Beige,0,Yes,97500,208.0,2.0,4.0,1.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188528,188528,Cadillac,Escalade ESV Platinum,2017,49000,Gasoline,Transmission w/Dual Shift Mode,White,Beige,0,Yes,27500,420.0,6.2,8.0,1.0,NaN
188529,188529,Mercedes-Benz,AMG C 43 AMG C 43 4MATIC,2018,28600,Gasoline,8-Speed A/T,White,Black,1,Yes,30000,385.0,3.0,6.0,1.0,8.0
188530,188530,Mercedes-Benz,AMG GLC 63 Base 4MATIC,2021,13650,Gasoline,7-Speed A/T,White,Black,0,Yes,86900,469.0,4.0,8.0,1.0,7.0
188531,188531,Audi,S5 3.0T Prestige,2022,13895,Gasoline,1-Speed Automatic,Daytona Gray Pearl Effect,Black,0,NaN,84900,NaN,3.0,NaN,1.0,1.0


In [ ]:
df = df.drop('transmission', axis=1)
df

,id,brand,model,model_year,milage,fuel_type,ext_col,int_col,accident,clean_title,price,power_hp,engine_volume_l,cylinders_count,is_automatic,gears_count
0,0,MINI,Cooper S Base,2007,213000,Gasoline,Yellow,Gray,0,Yes,4200,172.0,1.6,4.0,1.0,4.0
1,1,Lincoln,LS V8,2002,143250,Gasoline,Silver,Beige,1,Yes,4999,252.0,3.9,8.0,1.0,4.0
2,2,Chevrolet,Silverado 2500 LT,2002,136731,E85 Flex Fuel,Blue,Gray,0,Yes,13900,320.0,5.3,8.0,1.0,4.0
3,3,Genesis,G90 5.0 Ultimate,2017,19500,Gasoline,Black,Black,0,Yes,45000,420.0,5.0,8.0,1.0,NaN
4,4,Mercedes-Benz,Metris Base,2021,7388,Gasoline,Black,Beige,0,Yes,97500,208.0,2.0,4.0,1.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188528,188528,Cadillac,Escalade ESV Platinum,2017,49000,Gasoline,White,Beige,0,Yes,27500,420.0,6.2,8.0,1.0,NaN
188529,188529,Mercedes-Benz,AMG C 43 AMG C 43 4MATIC,2018,28600,Gasoline,White,Black,1,Yes,30000,385.0,3.0,6.0,1.0,8.0
188530,188530,Mercedes-Benz,AMG GLC 63 Base 4MATIC,2021,13650,Gasoline,White,Black,0,Yes,86900,469.0,4.0,8.0,1.0,7.0
188531,188531,Audi,S5 3.0T Prestige,2022,13895,Gasoline,Daytona Gray Pearl Effect,Black,0,NaN,84900,NaN,3.0,NaN,1.0,1.0


In [ ]:
df['model'].value_counts()

,count
model,
F-150 XLT,2945
M3 Base,2229
Camaro 2SS,1709
M4 Base,1622
Mustang GT Premium,1526
...,...
ForTwo Pure,2
XLR Base,1
X5 3.0i,1


In [ ]:
df['model'] = df['model'].str.split().str[0]




In [ ]:
df = df.drop('id', axis=1)
df

,brand,model,model_year,milage,fuel_type,ext_col,int_col,accident,clean_title,price,power_hp,engine_volume_l,cylinders_count,is_automatic,gears_count
0,MINI,Cooper,2007,213000,Gasoline,Yellow,Gray,0,Yes,4200,172.0,1.6,4.0,1.0,4.0
1,Lincoln,LS,2002,143250,Gasoline,Silver,Beige,1,Yes,4999,252.0,3.9,8.0,1.0,4.0
2,Chevrolet,Silverado,2002,136731,E85 Flex Fuel,Blue,Gray,0,Yes,13900,320.0,5.3,8.0,1.0,4.0
3,Genesis,G90,2017,19500,Gasoline,Black,Black,0,Yes,45000,420.0,5.0,8.0,1.0,NaN
4,Mercedes-Benz,Metris,2021,7388,Gasoline,Black,Beige,0,Yes,97500,208.0,2.0,4.0,1.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188528,Cadillac,Escalade,2017,49000,Gasoline,White,Beige,0,Yes,27500,420.0,6.2,8.0,1.0,NaN
188529,Mercedes-Benz,AMG,2018,28600,Gasoline,White,Black,1,Yes,30000,385.0,3.0,6.0,1.0,8.0
188530,Mercedes-Benz,AMG,2021,13650,Gasoline,White,Black,0,Yes,86900,469.0,4.0,8.0,1.0,7.0
188531,Audi,S5,2022,13895,Gasoline,Daytona Gray Pearl Effect,Black,0,NaN,84900,NaN,3.0,NaN,1.0,1.0


In [ ]:
df['clean_title'] = df['clean_title'].apply(lambda x: 1 if x == "Yes" else 0)
df

,brand,model,model_year,milage,fuel_type,ext_col,int_col,accident,clean_title,price,power_hp,engine_volume_l,cylinders_count,is_automatic,gears_count
0,MINI,Cooper,2007,213000,Gasoline,Yellow,Gray,0,1,4200,172.0,1.6,4.0,1.0,4.0
1,Lincoln,LS,2002,143250,Gasoline,Silver,Beige,1,1,4999,252.0,3.9,8.0,1.0,4.0
2,Chevrolet,Silverado,2002,136731,E85 Flex Fuel,Blue,Gray,0,1,13900,320.0,5.3,8.0,1.0,4.0
3,Genesis,G90,2017,19500,Gasoline,Black,Black,0,1,45000,420.0,5.0,8.0,1.0,NaN
4,Mercedes-Benz,Metris,2021,7388,Gasoline,Black,Beige,0,1,97500,208.0,2.0,4.0,1.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188528,Cadillac,Escalade,2017,49000,Gasoline,White,Beige,0,1,27500,420.0,6.2,8.0,1.0,NaN
188529,Mercedes-Benz,AMG,2018,28600,Gasoline,White,Black,1,1,30000,385.0,3.0,6.0,1.0,8.0
188530,Mercedes-Benz,AMG,2021,13650,Gasoline,White,Black,0,1,86900,469.0,4.0,8.0,1.0,7.0
188531,Audi,S5,2022,13895,Gasoline,Daytona Gray Pearl Effect,Black,0,0,84900,NaN,3.0,NaN,1.0,1.0


In [ ]:
base_colors = ['Black', 'White', 'Gray', 'Silver', 'Blue', 'Red',
               'Green', 'Gold', 'Brown', 'Orange', 'Beige', 'Yellow']

df['ext_col'] = df['ext_col'].apply(
    lambda x: next((c for c in base_colors if c.lower() in str(x).lower()), 'Other')
)

# Смотрим, сколько 'Other' получилось
print(f"Other цветов: {(df['ext_col'] == 'Other').sum()}")
print(df['ext_col'].value_counts())

Other цветов: 4522
ext_col
Black     53917
White     47945
Gray      26669
Silver    18809
Blue      15837
Red       11795
Other      4522
Green      2948
Gold       1668
Brown      1185
Orange     1148
Beige      1096
Yellow      994
Name: count, dtype: int64


In [ ]:
df

,brand,model,model_year,milage,fuel_type,ext_col,int_col,accident,clean_title,price,power_hp,engine_volume_l,cylinders_count,is_automatic,gears_count
0,MINI,Cooper,2007,213000,Gasoline,Yellow,Gray,0,1,4200,172.0,1.6,4.0,1.0,4.0
1,Lincoln,LS,2002,143250,Gasoline,Silver,Beige,1,1,4999,252.0,3.9,8.0,1.0,4.0
2,Chevrolet,Silverado,2002,136731,E85 Flex Fuel,Blue,Gray,0,1,13900,320.0,5.3,8.0,1.0,4.0
3,Genesis,G90,2017,19500,Gasoline,Black,Black,0,1,45000,420.0,5.0,8.0,1.0,NaN
4,Mercedes-Benz,Metris,2021,7388,Gasoline,Black,Beige,0,1,97500,208.0,2.0,4.0,1.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188528,Cadillac,Escalade,2017,49000,Gasoline,White,Beige,0,1,27500,420.0,6.2,8.0,1.0,NaN
188529,Mercedes-Benz,AMG,2018,28600,Gasoline,White,Black,1,1,30000,385.0,3.0,6.0,1.0,8.0
188530,Mercedes-Benz,AMG,2021,13650,Gasoline,White,Black,0,1,86900,469.0,4.0,8.0,1.0,7.0
188531,Audi,S5,2022,13895,Gasoline,Gray,Black,0,0,84900,NaN,3.0,NaN,1.0,1.0


In [ ]:
df.isna().sum()


,0
brand,0
model,0
model_year,0
milage,0
fuel_type,5083
ext_col,0
int_col,0
accident,0
clean_title,0
price,0


In [ ]:
df['fuel_type'] = df['fuel_type'].replace(['-', '–', '', 'None', 'none', 'NULL'], np.nan)
df['fuel_type'].value_counts()

,count
fuel_type,
Gasoline,165940
Hybrid,6832
E85 Flex Fuel,5406
Diesel,3955
Plug-In Hybrid,521
not supported,15


In [ ]:
X = df.drop('price', axis=1)
y = df['price']

In [ ]:
X.isna().sum()

,0
brand,0
model,0
model_year,0
milage,0
fuel_type,5864
ext_col,0
int_col,0
accident,0
clean_title,0
power_hp,33259


In [ ]:
categor_feature = X.select_dtypes(['object']).columns.to_list()
numerical_feature = X.select_dtypes([np.number]).columns.to_list()

for col in categor_feature:
  X[col] = X[col].fillna('Missing')

for col in numerical_feature:
    X[col] = X[col].fillna(
        X.groupby(['brand', 'model'])[col].transform('median')
    )

    X[col] = X[col].fillna(
        X.groupby('brand')[col].transform('median')
    )

    if X[col].isna().any():
        global_median = X[col].median()
        X[col] = X[col].fillna(global_median)


In [ ]:
X.isna().sum()

,0
brand,0
model,0
model_year,0
milage,0
fuel_type,0
ext_col,0
int_col,0
accident,0
clean_title,0
power_hp,0


,brand,model,model_year,milage,fuel_type,ext_col,int_col,accident,clean_title,power_hp,engine_volume_l,cylinders_count,is_automatic,gears_count
0,MINI,Cooper,2007,213000,Gasoline,Yellow,Gray,0,1,172.0,1.6,4.0,1.0,4.0
1,Lincoln,LS,2002,143250,Gasoline,Silver,Beige,1,1,252.0,3.9,8.0,1.0,4.0
2,Chevrolet,Silverado,2002,136731,E85 Flex Fuel,Blue,Gray,0,1,320.0,5.3,8.0,1.0,4.0
3,Genesis,G90,2017,19500,Gasoline,Black,Black,0,1,420.0,5.0,8.0,1.0,8.0
4,Mercedes-Benz,Metris,2021,7388,Gasoline,Black,Beige,0,1,208.0,2.0,4.0,1.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188528,Cadillac,Escalade,2017,49000,Gasoline,White,Beige,0,1,420.0,6.2,8.0,1.0,8.0
188529,Mercedes-Benz,AMG,2018,28600,Gasoline,White,Black,1,1,385.0,3.0,6.0,1.0,8.0
188530,Mercedes-Benz,AMG,2021,13650,Gasoline,White,Black,0,1,469.0,4.0,8.0,1.0,7.0
188531,Audi,S5,2022,13895,Gasoline,Gray,Black,0,0,333.0,3.0,6.0,1.0,1.0


In [ ]:
class Preprocessing:

  BASE_COLORS = ['Black', 'White', 'Gray', 'Silver', 'Blue', 'Red',
                   'Green', 'Gold', 'Brown', 'Orange', 'Beige', 'Yellow']

  FUEL_MAP = {
        'flex': 'Flex Fuel', 'flex fuel': 'Flex Fuel',
        'e85': 'Flex Fuel', 'ethanol': 'Flex Fuel',
        'petrol': 'Gasoline'
    }

  MISSING_VALUES = ['-', '–', '', 'None', 'none', 'NULL', 'N/A', 'na']

  def __init__(self, df:pd.DataFrame, is_linear: bool = True) -> None:
      self.df = df
      self.is_linear = is_linear

  def parse_engine(self, feature_str):

    power_match = re.search(r'(\d+\.?\d*)HP', feature_str)
    power = float(power_match.group(1)) if power_match else None

    displacement_match = re.search(r'(\d+\.?\d*)L', feature_str)
    displacement = float(displacement_match.group(1)) if displacement_match else None

    cylinders = None
    cylinders_match = re.search(r'(\d+)\s*Cylinder', feature_str)
    if cylinders_match:
        cylinders = int(cylinders_match.group(1))
    else:
        v_match = re.search(r'V(\d+)', feature_str)
        if v_match:
            cylinders = int(v_match.group(1))

    fuel_match = re.search(r'(Gasoline|Diesel|Electric|Petrol|Hybrid|Flex|Flex Fuel|E85|Ethanol)', feature_str, re.IGNORECASE)
    if fuel_match:
        fuel = fuel_match.group(1).capitalize()
        if fuel.lower() in ['Flex', 'Flex fuel', 'E85', 'Ethanol']:
            fuel = 'Flex Fuel'
        elif fuel.lower() == 'Petrol':
            fuel = 'Gasoline'
    else:
        fuel = None
    return pd.Series([power, displacement, cylinders, fuel])

  def parse_transmission(self, trans_str):
    if pd.isna(trans_str):
        return pd.Series([None, None])

    trans_str = str(trans_str)

    is_automatic = 1

    manual_keywords = ['M/T', 'Manual']
    if any(keyword in trans_str for keyword in manual_keywords):
        is_automatic = 0

    if 'CVT' in trans_str:
        is_automatic = 1

    speed_match = re.search(r'(\d+)-?\s*(Speed|A/T|M/T|Automatic)', trans_str)

    if speed_match:
        speeds = int(speed_match.group(1))
    elif 'A/T' in trans_str and 'Speed' not in trans_str:
        speeds = 4
    elif 'M/T' in trans_str and 'Speed' not in trans_str:
        speeds = 5
    elif 'Automatic' in trans_str and 'Speed' not in trans_str:
        speeds = 4
    elif 'CVT' in trans_str:
        speeds = 0
    else:
        speeds = None

    return pd.Series([is_automatic, speeds])


  def extract_color(self, name_feature: str):
      base_colors = ['Black', 'White', 'Gray', 'Silver', 'Blue', 'Red',
                'Green', 'Gold', 'Brown', 'Orange', 'Beige', 'Yellow']

      self.df[name_feature] = self.df[name_feature].apply(
          lambda x: next((c for c in base_colors if c.lower() in str(x).lower()), 'Other')
      )
      return self.df

  def detect_missing(self):
    categor_feature = self.df.select_dtypes(['object']).columns.to_list()
    numerical_feature = self.df.select_dtypes([np.number]).columns.to_list()

    for col in categor_feature:
      self.df[col] = self.df[col].fillna('Missing')

    for col in numerical_feature:
        self.df[col] = self.df[col].fillna(
            self.df.groupby(['brand', 'model'])[col].transform('median')
        )

        self.df[col] = self.df[col].fillna(
            self.df.groupby('brand')[col].transform('median')
        )

        if self.df[col].isna().any():
            global_median = self.df[col].median()
            self.df[col] = self.df[col].fillna(global_median)
    return self.df

  def make_preprocessing(self) -> pd.DataFrame:
    self.df = self.df.drop("id", axis=1)
    self.df[['power_hp', 'engine_volume_l', 'cylinders_count', 'type_fuel']] = self.df['engine'].apply(self.parse_engine)
    self.df = self.df.drop('engine', axis=1)
    self.df['accident'] = self.df['accident'].apply(lambda x: 1 if pd.notna(x) and x != "None reported" else 0)
    self.df['clean_title'] = self.df['clean_title'].apply(lambda x: 1 if x == 'Yes' else 0)
    self.df[['is_automatic', 'gears_count']] = self.df['transmission'].apply(self.parse_transmission)
    self.df = self.df.drop('transmission', axis=1)
    self.df['model'] = self.df['model'].str.split().str[0]
    self.df = self.extract_color('ext_col')
    self.df = self.extract_color('int_col')
    self.df['fuel_type'] = self.df['fuel_type'].replace(['-', '–', '', 'None', 'none', 'NULL'], np.nan)
    self.df = self.detect_missing()
    return self.df


In [ ]:
from optuna.pruners import HyperbandPruner
class Modeling:

  HYPERPARAMS = {}

  def __init__(self, model:CatBoostRegressor) -> None:
     self.model = model
     self.test_pred = None

  @staticmethod
  def show_metrics(y_pred, y_true):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)

    print(f"mae: {mae}")
    print(f"mse: {mse}")
    print(f"rmse: {rmse}")

  def find_hyperparams(self, df: pd.DataFrame, n_trials: int = 10 ) -> None:
    X = df.drop('price', axis=1)
    y = df['price']

    cat_features= df.select_dtypes(['object']).columns.to_list()

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

    def objective(trial: optuna.Trial) -> float:
      params = {
                'iterations':        trial.suggest_int('iterations', 500, 2000),
                'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
                'depth':             trial.suggest_int('depth', 4, 10),
                'l2_leaf_reg':       trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
                'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
                'random_strength':   trial.suggest_float('random_strength', 1e-2, 10.0, log=True),
                'border_count':      trial.suggest_int('border_count', 32, 255),
                'verbose': 100,
                'random_seed': 42,
                'loss_function': 'RMSE',
            }

      model = CatBoostRegressor(**params)
      model.fit(X_train, y_train, cat_features=cat_features, eval_set = (X_val, y_val), verbose=100)
      y_pred = model.predict(X_val)
      return mean_absolute_error(y_val, y_pred)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    self.HYPERPARAMS = study.best_params
    print(f"Best params: {self.HYPERPARAMS}")
    print(f"Best score: {study.best_value}")


  def model_fit_predict(self, df: pd.DataFrame, ) -> None:
    self.model = self.model(**self.HYPERPARAMS)
    X = df.drop('price', axis=1)
    cat_features = df.select_dtypes(['object']).columns.to_list()
    y = df['price']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

    if len(self.HYPERPARAMS) == 0:
      print("Подбор гиперпараметров ещё не реализован")
      self.model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), verbose=100)
      self.test_pred = self.model.predict(X_val)
      self.show_metrics(self.test_pred, y_val)
    else:
      self.model = self.model(**self.HYPERPARAMS)
      self.model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), verbose=100)
      self.test_pred = self.model.predict(X_val)
      self.show_metrics(self.test_pred, y_val)






In [ ]:
res_df = Preprocessing(pd.read_csv("/content/drive/MyDrive/model_price_car/train.csv"))
res_df = res_df.make_preprocessing()
res_df

,brand,model,model_year,milage,fuel_type,ext_col,int_col,accident,clean_title,price,power_hp,engine_volume_l,cylinders_count,type_fuel,is_automatic,gears_count
0,MINI,Cooper,2007,213000,Gasoline,Yellow,Gray,0,1,4200,172.0,1.6,4.0,Gasoline,1.0,4.0
1,Lincoln,LS,2002,143250,Gasoline,Silver,Beige,1,1,4999,252.0,3.9,8.0,Gasoline,1.0,4.0
2,Chevrolet,Silverado,2002,136731,E85 Flex Fuel,Blue,Gray,0,1,13900,320.0,5.3,8.0,Flex,1.0,4.0
3,Genesis,G90,2017,19500,Gasoline,Black,Black,0,1,45000,420.0,5.0,8.0,Gasoline,1.0,8.0
4,Mercedes-Benz,Metris,2021,7388,Gasoline,Black,Beige,0,1,97500,208.0,2.0,4.0,Gasoline,1.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188528,Cadillac,Escalade,2017,49000,Gasoline,White,Beige,0,1,27500,420.0,6.2,8.0,Gasoline,1.0,8.0
188529,Mercedes-Benz,AMG,2018,28600,Gasoline,White,Black,1,1,30000,385.0,3.0,6.0,Gasoline,1.0,8.0
188530,Mercedes-Benz,AMG,2021,13650,Gasoline,White,Black,0,1,86900,469.0,4.0,8.0,Gasoline,1.0,7.0
188531,Audi,S5,2022,13895,Gasoline,Gray,Black,0,0,84900,333.0,3.0,6.0,Missing,1.0,1.0


In [ ]:
model = Modeling(CatBoostRegressor())
model.find_hyperparams(res_df)
pred = model.model_fit_predict(res_df)
pred

[I 2026-04-29 15:50:13,319] A new study created in memory with name: no-name-6366216c-e15d-47b0-a559-e98388234a44


  0%|          | 0/10 [00:00<?, ?it/s]

0:	learn: 80222.9604104	test: 82357.2383611	best: 82357.2383611 (0)	total: 278ms	remaining: 4m 26s
100:	learn: 74776.4770043	test: 77239.8825201	best: 77239.8825201 (100)	total: 12.3s	remaining: 1m 44s
200:	learn: 74364.5555529	test: 77122.3520023	best: 77121.9374681 (199)	total: 21.6s	remaining: 1m 21s
300:	learn: 73987.2043225	test: 77087.3528569	best: 77083.5763069 (264)	total: 29.9s	remaining: 1m 5s
400:	learn: 73625.6052370	test: 77096.9512652	best: 77081.3747376 (320)	total: 40.1s	remaining: 55.9s
500:	learn: 73131.9032022	test: 77154.4873164	best: 77081.3747376 (320)	total: 48s	remaining: 44s
600:	learn: 72643.8472455	test: 77216.3092019	best: 77081.3747376 (320)	total: 58s	remaining: 34.7s
700:	learn: 72314.2808427	test: 77234.0907308	best: 77081.3747376 (320)	total: 1m 8s	remaining: 25.2s
800:	learn: 72079.2712239	test: 77284.6171874	best: 77081.3747376 (320)	total: 1m 16s	remaining: 15.1s
900:	learn: 71791.8790210	test: 77351.5437350	best: 77081.3747376 (320)	total: 1m 26s	re

TypeError: 'CatBoostRegressor' object is not callable